# CosyVoice2 Input 4: Phrase ID and Cache Path Generator

This Colab notebook computes the `input4` phrase IDs and expected public cache paths for the Dicta web app.

It uses the same stable hashing logic as the app so generated phrase IDs match cached file names in `/public/tts-cache/cosyvoice/{language}/{hash}.wav`.


In [ ]:
# Section 1: Import Required Libraries
import json
from pathlib import Path

print("Libraries imported successfully")


In [ ]:
# Section 2: Set Up Input Value
INPUT_MODE = 4
LANGUAGE = "en"
SAMPLE_PHRASES = [
    "Welcome to Dicta, your adaptive dictation trainer.",
    "Please pronounce each phrase clearly and keep your pace steady.",
]

print(f"Input mode: {INPUT_MODE}")
print(f"Language: {LANGUAGE}")
print(f"Sample phrases: {len(SAMPLE_PHRASES)}")


In [ ]:
# Section 3: Define Processing Function

def normalize_phrase(value: str) -> str:
    return " ".join(value.strip().lower().split())


def stable_phrase_hash(value: str) -> str:
    # Matches the app's TS stablePhraseHash implementation.
    normalized = normalize_phrase(value)
    h = 2166136261
    for ch in normalized:
        h ^= ord(ch)
        h = (h + (h << 1) + (h << 4) + (h << 7) + (h << 8) + (h << 24)) & 0xFFFFFFFF
    return f"{h:08x}"


def build_qwen_cloud_phrase_id(language: str, phrase_text: str) -> str:
    composed = f"{language}:{normalize_phrase(phrase_text)}"
    return f"{language}:{stable_phrase_hash(composed)}"


def qwen_cloud_public_url(phrase_id: str) -> str:
    language, digest = phrase_id.split(":", 1)
    return f"/tts-cache/cosyvoice/{language}/{digest}.wav"


def qwen_cloud_local_path(phrase_id: str) -> Path:
    language, digest = phrase_id.split(":", 1)
    return Path("public") / "tts-cache" / "cosyvoice" / language / f"{digest}.wav"


def build_manifest_row(language: str, phrase_text: str) -> dict:
    phrase_id = build_qwen_cloud_phrase_id(language, phrase_text)
    return {
        "phraseId": phrase_id,
        "language": language,
        "text": phrase_text,
        "audioUrl": qwen_cloud_public_url(phrase_id),
    }

print("Processing functions defined")


In [ ]:
# Section 4: Run Logic with Input 4
manifest_entries = [build_manifest_row(LANGUAGE, phrase) for phrase in SAMPLE_PHRASES]

for entry in manifest_entries:
    print(f"Phrase text: {entry['text']}")
    print(f"Phrase ID: {entry['phraseId']}")
    print(f"Public URL: {entry['audioUrl']}")
    print(f"Local path: {qwen_cloud_local_path(entry['phraseId'])}")
    print("---")

NOTE = (
    "If you produce cached WAV files in Colab, save them under the local path shown above. "
    "Then the app can load them from /tts-cache/cosyvoice/{language}/{hash}.wav."
)
print(NOTE)


In [ ]:
# Section 5: Display Output and Verify Results
output = {
    "inputMode": INPUT_MODE,
    "language": LANGUAGE,
    "phraseCount": len(manifest_entries),
    "manifest": manifest_entries,
}

print(json.dumps(output, indent=2, ensure_ascii=False))

# Optional: write a manifest file for use in the repo
output_path = Path("qwen_cloud_input4_manifest.json")
output_path.write_text(json.dumps(output, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Manifest written to {output_path.resolve()}")


## Section 6: Paste Manifest JSON From Dicta

In Dicta (Input #4), use **Copy Cache Manifest JSON** or **Export Cache Manifest JSON**.

Paste the JSON below or upload a file named `qwen-cache-manifest-*.json`.


In [ ]:
MANIFEST_JSON = r"""
{
  "engine": "qwen-cloud",
  "language": "en",
  "phrases": []
}
""".strip()

manifest = json.loads(MANIFEST_JSON)
print("Loaded manifest:", manifest.get("engine"), manifest.get("language"), "phrases=", len(manifest.get("phrases", [])))


## Section 7: CosyVoice/CosyVoice2 Setup (Free)

This notebook is designed to be *free* by running an open-source TTS model in Colab.

CosyVoice2 is commonly used from the upstream repo. If your collaborator already has a working setup, replace the install and `synthesize_wav(...)` function below with the exact working code.


In [ ]:
# If you already have CosyVoice2 code working in your Colab, you can skip this cell.
# The exact install steps may differ by checkpoint.

!rm -rf CosyVoice
!git clone --depth 1 https://github.com/FunAudioLLM/CosyVoice.git
!pip -q install -r CosyVoice/requirements.txt
!pip -q install soundfile

print("CosyVoice repo cloned and requirements installed.")


In [ ]:
import numpy as np
import soundfile as sf
from typing import Tuple

# TODO: Replace this with your collaborator's working CosyVoice2 inference function.
# This placeholder produces silence and will NOT be usable for dictation.

def synthesize_wav(text: str, language: str) -> Tuple[np.ndarray, int]:
    """Return (waveform, sample_rate). Replace with CosyVoice/CosyVoice2 inference."""
    sr = 24000
    seconds = max(1.0, min(8.0, len(text) / 20.0))
    wave = np.zeros(int(sr * seconds), dtype=np.float32)
    return wave, sr

print("synthesize_wav placeholder ready (replace with real CosyVoice2 inference).")


## Section 8: Generate Cache WAVs + Manifest + Zip

Writes to the repo-mirrored folder:

`public/tts-cache/cosyvoice/{language}/{hash}.wav`

Then zips `public/tts-cache/cosyvoice/` so you can download and extract it into your local Dicta repo.


In [ ]:
from datetime import datetime
import zipfile

out_root = Path("public") / "tts-cache" / "cosyvoice"
phrases = manifest.get("phrases", [])
if not phrases:
    raise ValueError("Manifest has no phrases. Paste a real manifest from Dicta.")

for entry in phrases:
    phrase_id = entry["id"]
    language, digest = phrase_id.split(":", 1)
    text = entry.get("text", "")
    wav_path = out_root / language / f"{digest}.wav"
    wav_path.parent.mkdir(parents=True, exist_ok=True)

    wave, sr = synthesize_wav(text, language)
    sf.write(str(wav_path), wave, sr)

    # Update manifest with duration if we can infer it
    entry["durationMs"] = int(1000 * (len(wave) / float(sr))) if sr else None
    entry["audioUrl"] = f"/tts-cache/cosyvoice/{language}/{digest}.wav"

# Write per-language manifest.json for the app's optional manifest loader
language_groups = {}
for entry in phrases:
    lang = entry["language"]
    language_groups.setdefault(lang, []).append(entry)

for lang, items in language_groups.items():
    lang_manifest = {
        "engine": "qwen-cloud",
        "language": lang,
        "phrases": items,
    }
    lang_dir = out_root / lang
    lang_dir.mkdir(parents=True, exist_ok=True)
    (lang_dir / "manifest.json").write_text(json.dumps(lang_manifest, indent=2, ensure_ascii=False), encoding="utf-8")

# Zip for download
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
zip_path = Path(f"qwen-cache-{stamp}.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in out_root.rglob("*"):
        if path.is_file():
            zf.write(path, path)

print("Wrote cache under:", out_root)
print("Zip ready:", zip_path.resolve())
